# Task 3 (Bagian Anggota 1) — Image EDA, Preprocessing, Representation Learning & Image Embedding

**Notebook:** `task3_anggota1_gambar.ipynb` — dijalankan **lokal**, *Run All*.
**Keluaran utama:** `embedding_gambar.npz` (kunci `id_sampel`, `embedding`) yang dimuat oleh `task3_multimodal_clustering.ipynb` (Section 6, Anggota 2).

| Tugas | Section |
|---|---|
| EDA gambar mikroskopis | 3 |
| Preprocessing gambar | 4 |
| Representation learning gambar (PCA, Autoencoder, Contrastive) | 5–6 |
| Evaluasi representasi tanpa label & membuat embedding | 7–8 |

**Aturan:** label jenis material **tidak ada dan tidak dipakai**. Kolom `mikroskop` (M1/M2) dipakai hanya sebagai *diagnostik domain shift*, bukan sebagai label kelas.

# SECTION 1 — Library & Konfigurasi

In [ ]:
import os, re, sys, time, random, warnings, zipfile
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import silhouette_score, normalized_mutual_info_score
from scipy import ndimage

warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 100
SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------- KONFIGURASI ----------------
CANDIDATE_DIRS = ["dataset", "../dataset", "."]
NPZ_NAMES, META_FILE = ["clustering.npz"], "clustering_meta.csv"
OUTPUT_DIR = Path("outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
RETRAIN = False            # True = abaikan cache bobot dan latih ulang
EMB_DIM = 32               # dimensi embedding gambar
EPOCHS_CL, EPOCHS_AE = 12, 8       # CPU: lihat waktu per epoch di Section 6; GPU: boleh 40 / 20
BATCH, LR, TAU = 256, 1e-3, 0.2
# Batas rentang augmentasi (disesuaikan dengan temuan EDA di Section 3):
ROT_MAX_DEG = 180          # rotasi acak +-180 derajat  (set 0 jika EDA menunjukkan rotasi BUKAN distorsi)
SCALE_RANGE = (0.8, 1.25)  # zoom acak
SHIFT_FRAC  = 0.08         # translasi acak (fraksi ukuran gambar)
OUT_EMB_FILE = "embedding_gambar.npz"
print("Device:", DEVICE, "| torch", torch.__version__)

# SECTION 2 — Memuat Data

In [ ]:
def find_data():
    for d in CANDIDATE_DIRS:
        for n in NPZ_NAMES:
            if (Path(d)/n).exists() and (Path(d)/META_FILE).exists(): return Path(d), n
    return None, None
DATA_DIR, NPZ_NAME = find_data()
assert DATA_DIR is not None, f"{NPZ_NAMES}+{META_FILE} tidak ditemukan di {CANDIDATE_DIRS}"
meta = pd.read_csv(DATA_DIR/META_FILE)
npz = np.load(DATA_DIR/NPZ_NAME)
GAMBAR = npz["gambar"]                                   # (N, 64, 64) uint8
N = len(meta); IDS = meta["id_sampel"].astype(str).to_numpy()
MIC = (meta["mikroskop"].to_numpy() == "M2").astype(int)  # M1=0, M2=1
assert GAMBAR.shape[0] == N and meta["id_sampel"].is_unique
assert (meta["indeks"].to_numpy() == np.arange(N)).all(), "baris i meta harus = gambar[i]"
print("gambar:", GAMBAR.shape, GAMBAR.dtype, "| M1:", int((MIC==0).sum()), "M2:", int((MIC==1).sum()))
print("NaN:", int(np.isnan(GAMBAR.astype(np.float32)).sum()), "| min/max:", GAMBAR.min(), GAMBAR.max())

# SECTION 3 — Image EDA

Pertanyaan yang dijawab (semua tanpa label): bagaimana rupa gambar, apakah ada perbedaan **mikroskop** (kecerahan, kontras, ketajaman, derau), dan apakah ada **distorsi geometri** (rotasi, skala) yang harus dibuat *tidak berpengaruh* pada embedding.

In [ ]:
# 3.1 Contoh gambar per mikroskop
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(4, 10, figsize=(15, 6.4))
for r, (nm, v) in enumerate([("M1", 0), ("M2", 1)]):
    pick = rng.choice(np.where(MIC == v)[0], 20, replace=False)
    for j, i in enumerate(pick):
        ax = axes[2*r + j//10, j%10]; ax.imshow(GAMBAR[i], cmap="gray", vmin=0, vmax=255); ax.axis("off"); ax.set_title(f"{nm} #{i}", fontsize=7)
plt.suptitle("Atas: M1 | Bawah: M2"); plt.tight_layout(); plt.show()

In [ ]:
# 3.2 Statistik tingkat gambar per mikroskop: kecerahan, kontras, ketajaman (variansi Laplacian), derau (MAD residu median-filter)
X = GAMBAR.astype(np.float32)
f_mean, f_std = X.mean((1,2)), X.std((1,2))
lap = X[:, 1:-1, 1:-1]*4 - X[:, :-2, 1:-1] - X[:, 2:, 1:-1] - X[:, 1:-1, :-2] - X[:, 1:-1, 2:]
f_sharp = lap.var((1,2))
sub = rng.choice(N, min(4000, N), replace=False)
f_noise = np.zeros(N, np.float32)
f_noise[sub] = [np.median(np.abs(X[i] - ndimage.median_filter(X[i], 3))) for i in sub]
STAT = pd.DataFrame({"kecerahan": f_mean, "kontras": f_std, "ketajaman": f_sharp, "derau": np.where(f_noise > 0, f_noise, np.nan), "mikroskop": np.where(MIC==1, "M2", "M1")})
print(STAT.groupby("mikroskop").agg(["mean", "std"]).round(2).T)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
for k, col in enumerate(["kecerahan", "kontras", "ketajaman"]):  # (derau hanya dihitung pada subsampel; lihat tabel)
    for nm in ["M1", "M2"]: ax[k].hist(STAT.loc[STAT.mikroskop==nm, col], bins=60, alpha=.6, label=nm)
    ax[k].set_title(col); ax[k].legend()
plt.tight_layout(); plt.show()

In [ ]:
# 3.3 Histogram intensitas piksel & spektrum frekuensi radial (apakah satu mikroskop lebih buram / lebih berderau?)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for v, nm in [(0, "M1"), (1, "M2")]:
    idx = rng.choice(np.where(MIC==v)[0], 3000, replace=False)
    ax[0].hist(GAMBAR[idx].ravel(), bins=64, alpha=.6, density=True, label=nm)
    P = np.abs(np.fft.fftshift(np.fft.fft2(X[idx] - X[idx].mean((1,2), keepdims=True)), axes=(1,2)))**2
    yy, xx = np.indices((64, 64)); rr = np.hypot(yy-32, xx-32).astype(int).ravel()
    prof = np.bincount(rr, P.mean(0).ravel())[:32] / np.bincount(rr)[:32]
    ax[1].semilogy(prof, label=nm)
ax[0].set(title="Histogram intensitas piksel", xlabel="intensitas"); ax[0].legend()
ax[1].set(title="Spektrum daya radial (frekuensi rendah -> tinggi)", xlabel="radius frekuensi"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# 3.4 Orientasi dominan per gambar (structure tensor). Rotasi adalah distorsi atau penanda kelas?
def dominant_orientation(img):
    gy, gx = np.gradient(ndimage.gaussian_filter(img.astype(np.float32), 1.0))
    jxx, jyy, jxy = (gx*gx).sum(), (gy*gy).sum(), (gx*gy).sum()
    coherence = np.hypot(jxx - jyy, 2*jxy) / (jxx + jyy + 1e-9)
    return 0.5*np.degrees(np.arctan2(2*jxy, jxx - jyy)), coherence
ori = np.array([dominant_orientation(X[i]) for i in sub])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].hist(ori[:, 0], bins=36); ax[0].set(title="Sudut dominan (derajat)", xlabel="derajat")
ax[1].hist(ori[:, 1], bins=40); ax[1].set(title="Koherensi orientasi (0=isotropik, 1=sangat berarah)")
plt.tight_layout(); plt.show()
print(f"Koherensi rata-rata = {ori[:,1].mean():.3f}")

In [ ]:
# 3.5 Struktur utama data mentah: apakah PC1 hanya mengikuti kecerahan / mikroskop (domain shift)?
Xf = X.reshape(N, -1)
pca_raw = PCA(10, random_state=SEED).fit(Xf[sub]); S = pca_raw.transform(Xf[sub])
print("Varians PC1..5:", pca_raw.explained_variance_ratio_[:5].round(3))
print("|korelasi| PC1 vs kecerahan :", abs(np.corrcoef(S[:,0], f_mean[sub])[0,1]).round(3))
print("|korelasi| PC1 vs mikroskop :", abs(np.corrcoef(S[:,0], MIC[sub])[0,1]).round(3))
km = KMeans(4, n_init=5, random_state=SEED).fit_predict(StandardScaler().fit_transform(S))
print("NMI(KMeans k=4 pada PCA mentah, mikroskop) =", round(normalized_mutual_info_score(km, MIC[sub]), 3), "(tinggi = cluster mengikuti mikroskop, bukan material)")

**Interpretasi EDA (ISI SETELAH MELIHAT HASIL DI ATAS).** Jawab singkat, masing-masing satu-dua kalimat dengan angka dari output:
1. Perbedaan **kecerahan/kontras/ketajaman/derau** antara M1 dan M2: ...
2. Apakah **rotasi** gambar tampak acak (histogram sudut rata) → distorsi, atau terkonsentrasi → bisa jadi penanda kelas: ...
3. Apakah PC1 data mentah mengikuti **kecerahan/mikroskop** (korelasi tinggi → domain shift): ...
4. **Konsekuensi untuk preprocessing:** normalisasi per gambar (Section 4); bila rotasi terbukti acak → augmentasi rotasi penuh (`ROT_MAX_DEG=180`), bila tidak → `ROT_MAX_DEG=0`.

# SECTION 4 — Preprocessing Gambar

Semua langkah **per gambar** (tanpa label, tanpa statistik global yang di-*fit*), sehingga tidak ada kebocoran:
1. Skala ke [0, 1] (`uint8 / 255`).
2. **Standardisasi per gambar** (kurangi rata-rata, bagi std): menghapus selisih kecerahan/kontras antar-mikroskop (domain shift).
3. (Opsional) *clip* ke ±4 std agar piksel ekstrem tidak mendominasi.
Augmentasi (rotasi, skala, translasi, gamma, derau) hanya dipakai **saat training** representation learning (Section 5).

In [ ]:
def preprocess(U8):
    x = U8.astype(np.float32) / 255.0
    m, s = x.mean((1,2), keepdims=True), x.std((1,2), keepdims=True)
    return np.clip((x - m) / (s + 1e-6), -4, 4).astype(np.float32)
XP = preprocess(GAMBAR)                                      # (N, 64, 64)
fig, ax = plt.subplots(2, 6, figsize=(15, 5))
for c, i in enumerate(rng.choice(N, 6, replace=False)):
    ax[0, c].imshow(GAMBAR[i], cmap="gray", vmin=0, vmax=255); ax[0, c].set_title(f"mentah #{i} ({'M2' if MIC[i] else 'M1'})", fontsize=8)
    ax[1, c].imshow(XP[i], cmap="gray"); ax[1, c].set_title("setelah normalisasi", fontsize=8)
    ax[0, c].axis("off"); ax[1, c].axis("off")
plt.tight_layout(); plt.show()
pp = pd.DataFrame({"kecerahan": XP.mean((1,2)), "kontras": XP.std((1,2)), "mikroskop": np.where(MIC==1, "M2", "M1")})
print(pp.groupby("mikroskop").mean().round(3))
print("Tensor:", XP.shape, XP.dtype, "| NMI(KMeans k=4 pada PCA hasil preprocessing, mikroskop):",
      round(normalized_mutual_info_score(KMeans(4, n_init=5, random_state=SEED).fit_predict(StandardScaler().fit_transform(PCA(10, random_state=SEED).fit_transform(XP.reshape(N,-1)[sub]))), MIC[sub]), 3))

**Interpretasi.** Setelah normalisasi, kecerahan dan kontras kedua mikroskop sama (0 dan 1) sehingga selisih kecerahan hilang. Bandingkan NMI-mikroskop di atas dengan NMI di 3.5: bila turun, domain shift berkurang. *(Tulis angkanya di sini.)*

# SECTION 5 — Image Representation Learning

Tiga representasi dibandingkan (semua **tanpa label**):

| Metode | Ide | Catatan |
|---|---|---|
| **PCA** (baseline non-DL) | proyeksi linear pada gambar ternormalisasi | pembanding; peka rotasi |
| **Convolutional Autoencoder** | latent 32-d yang merekonstruksi gambar | menyimpan semua variasi, termasuk distorsi |
| **Contrastive CNN (SimCLR/NT-Xent)** | dua *view* augmentasi dari gambar yang sama harus berdekatan | belajar **invarian** terhadap rotasi/skala/translasi/kecerahan → fokus pada tekstur mikrostruktur |

In [ ]:
class ConvBlock(nn.Sequential):        # satu conv + BN + ReLU + pool (ringan agar cukup cepat di CPU)
    def __init__(s, i, o): super().__init__(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True), nn.MaxPool2d(2))
class ImgEncoder(nn.Module):                 # 64 -> 32 -> 16 -> 8 -> 4 -> GAP -> embedding
    def __init__(s, emb=EMB_DIM, ch=(16, 32, 64, 128)):
        super().__init__(); L, i = [], 1
        for o in ch: L.append(ConvBlock(i, o)); i = o
        s.conv = nn.Sequential(*L); s.fc = nn.Linear(ch[-1], emb)
    def forward(s, x): return s.fc(s.conv(x).mean((2, 3)))
class ContrastiveModel(nn.Module):
    def __init__(s): super().__init__(); s.enc = ImgEncoder(); s.head = nn.Sequential(nn.Linear(EMB_DIM, 64), nn.ReLU(True), nn.Linear(64, 32))
    def forward(s, x): return F.normalize(s.head(s.enc(x)), dim=1)
class ConvAE(nn.Module):
    def __init__(s):
        super().__init__(); s.enc = ImgEncoder()
        s.fc = nn.Linear(EMB_DIM, 128 * 4 * 4)
        def up(i, o): return nn.Sequential(nn.ConvTranspose2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.ReLU(True))
        s.dec = nn.Sequential(up(128, 96), up(96, 48), up(48, 24), up(24, 12), nn.Conv2d(12, 1, 3, padding=1))   # 4 -> 64
    def forward(s, x): z = s.enc(x); return s.dec(F.relu(s.fc(z)).view(-1, 128, 4, 4)), z

def augment(x, rot=ROT_MAX_DEG, scale=SCALE_RANGE, shift=SHIFT_FRAC):
    """x: (B,1,64,64) hasil preprocessing. Rotasi, skala, translasi, flip, gamma-like, derau; normalisasi per gambar ulang."""
    B = x.shape[0]; dev = x.device
    a = (torch.rand(B, device=dev)*2 - 1) * np.deg2rad(rot)
    sc = torch.empty(B, device=dev).uniform_(*scale)
    flip = (torch.rand(B, device=dev) < 0.5).float()*2 - 1
    th = torch.zeros(B, 2, 3, device=dev)
    th[:, 0, 0] = torch.cos(a)/sc*flip; th[:, 0, 1] = -torch.sin(a)/sc; th[:, 1, 0] = torch.sin(a)/sc*flip; th[:, 1, 1] = torch.cos(a)/sc
    th[:, :, 2] = (torch.rand(B, 2, device=dev)*2 - 1) * shift * 2
    y = F.grid_sample(x, F.affine_grid(th, x.shape, align_corners=False), padding_mode="reflection", align_corners=False)
    y = y * torch.empty(B, 1, 1, 1, device=dev).uniform_(0.7, 1.4) + torch.empty(B, 1, 1, 1, device=dev).uniform_(-0.5, 0.5)   # kontras/kecerahan
    y = y + 0.1 * torch.randn_like(y)
    return (y - y.mean((1, 2, 3), keepdim=True)) / (y.std((1, 2, 3), keepdim=True) + 1e-6)

def nt_xent(z1, z2, tau=TAU):
    z = torch.cat([z1, z2]); n = z1.shape[0]
    sim = z @ z.T / tau; sim.fill_diagonal_(-1e9)
    tgt = torch.cat([torch.arange(n, 2*n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, tgt)

XT = torch.from_numpy(XP[:, None])          # CPU tensor (N,1,64,64)
@torch.no_grad()
def embed(enc, bs=1024):
    enc.eval(); out = [enc(XT[i:i+bs].to(DEVICE)).cpu() for i in range(0, N, bs)]
    return torch.cat(out).numpy().astype(np.float32)

def fit_or_load(name, model, train_fn):
    p = OUTPUT_DIR / f"{name}.pt"; model = model.to(DEVICE)
    if p.exists() and not RETRAIN:
        ck = torch.load(p, map_location=DEVICE); model.load_state_dict(ck["state"]); print(f"[cache] {name} dimuat dari {p}"); return model, ck["hist"]
    hist = train_fn(model); torch.save({"state": model.state_dict(), "hist": hist}, p); return model, hist

def train_contrastive(m):
    opt = torch.optim.AdamW(m.parameters(), LR, weight_decay=1e-4); steps = EPOCHS_CL * (N // BATCH)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, LR*3, total_steps=steps); hist = []; t0 = time.time()
    for ep in range(EPOCHS_CL):
        m.train(); perm = torch.randperm(N); tot = 0
        for i in range(N // BATCH):
            xb = XT[perm[i*BATCH:(i+1)*BATCH]].to(DEVICE)
            loss = nt_xent(m(augment(xb)), m(augment(xb)))
            opt.zero_grad(); loss.backward(); opt.step(); sch.step(); tot += loss.item()
        hist.append(tot / (N // BATCH)); print(f"CL epoch {ep+1}/{EPOCHS_CL} loss {hist[-1]:.4f} ({time.time()-t0:.0f}s)")
    return hist
def train_ae(m):
    opt = torch.optim.AdamW(m.parameters(), LR, weight_decay=1e-4); hist = []; t0 = time.time()
    for ep in range(EPOCHS_AE):
        m.train(); perm = torch.randperm(N); tot = 0
        for i in range(N // BATCH):
            xb = XT[perm[i*BATCH:(i+1)*BATCH]].to(DEVICE); rec, _ = m(xb); loss = F.mse_loss(rec, xb)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        hist.append(tot / (N // BATCH)); print(f"AE epoch {ep+1}/{EPOCHS_AE} MSE {hist[-1]:.4f} ({time.time()-t0:.0f}s)")
    return hist

In [ ]:
# Contoh augmentasi (dua view dari gambar yang sama) — periksa bahwa tekstur tetap terlihat
xb = XT[:6].to(DEVICE); fig, ax = plt.subplots(3, 6, figsize=(13, 6.6))
v1, v2 = augment(xb).cpu(), augment(xb).cpu()
for c in range(6):
    for r, (im, t) in enumerate([(xb[c].cpu(), "asli"), (v1[c], "view 1"), (v2[c], "view 2")]):
        ax[r, c].imshow(im[0], cmap="gray"); ax[r, c].set_title(t, fontsize=8); ax[r, c].axis("off")
plt.tight_layout(); plt.show()

# SECTION 6 — Pelatihan (atau muat dari cache `outputs/`)

In [ ]:
set_seed(); cl_model, HIST_CL = fit_or_load("img_contrastive", ContrastiveModel(), train_contrastive)
set_seed(); ae_model, HIST_AE = fit_or_load("img_autoencoder", ConvAE(), train_ae)
EMB = {"PCA": PCA(EMB_DIM, random_state=SEED).fit_transform(XP.reshape(N, -1)).astype(np.float32),
       "Autoencoder": embed(ae_model.enc), "Contrastive": embed(cl_model.enc)}
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(HIST_CL, c="#c44e52"); ax[0].set(title="Contrastive: NT-Xent", xlabel="epoch")
ax[1].plot(HIST_AE); ax[1].set(title="Autoencoder: MSE rekonstruksi", xlabel="epoch", yscale="log")
plt.tight_layout(); plt.show()
for k, v in EMB.items(): print(f"{k:12s} {v.shape} finite={np.isfinite(v).all()}")

# SECTION 7 — Evaluasi Representasi (tanpa label)

| Ukuran | Arti | Arah baik |
|---|---|---|
| **Silhouette terbaik (k=2..8)** | keterpisahan kelompok alami pada embedding | tinggi |
| **Kebocoran mikroskop** (*balanced accuracy* regresi logistik, 5-fold; 0,5 = acak) | seberapa mudah embedding ditebak berasal dari M1/M2 → domain shift masih tersimpan | rendah |
| **Invariansi** (cosine similarity antara embedding gambar dan versi teraugmentasinya, dikurangi baseline pasangan acak) | seberapa tahan embedding terhadap rotasi/skala/derau | tinggi |

In [ ]:
SUB = np.random.default_rng(SEED).choice(N, min(8000, N), replace=False)
def best_silhouette(E):
    Z = PCA(min(16, E.shape[1]), random_state=SEED).fit_transform(StandardScaler().fit_transform(E[SUB])); best = (-1, None)
    for k in range(2, 9):
        s = silhouette_score(Z, KMeans(k, n_init=5, random_state=SEED).fit_predict(Z), sample_size=min(4000, len(Z)), random_state=SEED)
        if s > best[0]: best = (s, k)
    return best
def mic_leak(E):
    Z = StandardScaler().fit_transform(E[SUB]); return cross_val_score(LogisticRegression(max_iter=500, class_weight="balanced"), Z, MIC[SUB], cv=5, scoring="balanced_accuracy").mean()
@torch.no_grad()
def invariance(enc):
    enc.eval(); x = XT[SUB[:2000]].to(DEVICE); set_seed()
    a, b = F.normalize(enc(x), dim=1), F.normalize(enc(augment(x)), dim=1)
    return (a * b).sum(1).mean().item() - (a * b.roll(1, 0)).sum(1).mean().item()
def pca_invariance():          # PCA: proyeksikan versi teraugmentasi dengan PCA yang sama
    p = PCA(EMB_DIM, random_state=SEED).fit(XP.reshape(N, -1)[SUB]); x = XT[SUB[:2000]]; set_seed()
    a = F.normalize(torch.from_numpy(p.transform(x.reshape(len(x), -1).numpy())), dim=1)
    b = F.normalize(torch.from_numpy(p.transform(augment(x).reshape(len(x), -1).numpy())), dim=1)
    return (a * b).sum(1).mean().item() - (a * b.roll(1, 0)).sum(1).mean().item()
INV = {"PCA": pca_invariance(), "Autoencoder": invariance(ae_model.enc), "Contrastive": invariance(cl_model.enc)}
rows = []
for k, E in EMB.items():
    s, kk = best_silhouette(E); rows.append({"metode": k, "silhouette": s, "pada k": kk, "kebocoran mikroskop (bal.acc)": mic_leak(E), "invariansi": INV[k]})
EVAL = pd.DataFrame(rows).set_index("metode"); print(EVAL.round(3))

In [ ]:
# Pemilihan: agregasi peringkat (1 = terbaik); seri -> pilih Contrastive/ AE di atas PCA. Bisa di-override manual lewat IMG_CHOICE.
R = pd.DataFrame({"sil": EVAL["silhouette"].rank(ascending=False), "bocor": EVAL["kebocoran mikroskop (bal.acc)"].rank(),
                  "inv": EVAL["invariansi"].rank(ascending=False)}); print(R.assign(rata2=R.mean(1)).round(2))
IMG_CHOICE = R.mean(1).idxmin()          # ganti manual mis. "Contrastive" bila perlu, dan jelaskan alasannya di interpretasi
print("Representasi gambar terpilih:", IMG_CHOICE)

**Interpretasi.** *(Isi setelah melihat tabel.)* Contoh kerangka: "Contrastive memiliki invariansi tertinggi (…) dan kebocoran mikroskop terendah (…), sehingga embedding-nya lebih mencerminkan tekstur material daripada perbedaan mikroskop/rotasi. PCA/AE memiliki silhouette (…) tetapi masih menyimpan informasi mikroskop (bal.acc …)."
**Catatan jujur:** semua ukuran di atas tanpa label sehingga bukan jaminan ARI tinggi; keputusan akhir berada pada evaluasi multimodal oleh Anggota 2 (ARI resmi dihitung dosen).

In [ ]:
# Visualisasi: UMAP embedding terpilih, diwarnai mikroskop (idealnya M1 dan M2 bercampur)
import umap
U = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=SEED).fit_transform(StandardScaler().fit_transform(EMB[IMG_CHOICE][SUB]))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
ax[0].scatter(*U.T, c=MIC[SUB], cmap="coolwarm", s=3, alpha=.6); ax[0].set_title(f"UMAP {IMG_CHOICE} — warna: mikroskop (merah=M2)")
ax[1].scatter(*U.T, c=f_mean[SUB], cmap="viridis", s=3, alpha=.6); ax[1].set_title("warna: kecerahan mentah")
plt.tight_layout(); plt.show()

# SECTION 8 — Membuat & Mengekspor Image Embedding

Format sesuai loader Anggota 2 (Section 6 notebook clustering): `id_sampel` (string, N) + `embedding` (float32, N×D).

In [ ]:
E_OUT = EMB[IMG_CHOICE].astype(np.float32)
assert E_OUT.shape == (N, EMB_DIM) and np.isfinite(E_OUT).all()
np.savez(OUTPUT_DIR / OUT_EMB_FILE, id_sampel=IDS.astype(str), embedding=E_OUT)
# Simpan semua kandidat juga (opsional untuk perbandingan oleh Anggota 2)
np.savez(OUTPUT_DIR / "embedding_gambar_semua.npz", id_sampel=IDS.astype(str), **{k.lower(): v.astype(np.float32) for k, v in EMB.items()})
# Verifikasi: baca ulang seperti loader Anggota 2
z = np.load(OUTPUT_DIR / OUT_EMB_FILE, allow_pickle=False)
assert set(z.files) == {"id_sampel", "embedding"} and (z["id_sampel"] == IDS).all() and z["embedding"].shape == (N, EMB_DIM)
print(f"OK -> {(OUTPUT_DIR / OUT_EMB_FILE).resolve()} | embedding {z['embedding'].shape} {z['embedding'].dtype} | metode: {IMG_CHOICE}")
print("Langkah berikutnya: salin file ini ke folder dataset/ (di samping clustering_meta.csv) dan kirim ke Anggota 2 / commit ke GitHub.")

# Kesimpulan (Anggota 1)
*(Isi 4–6 kalimat: temuan EDA utama → keputusan preprocessing → metode representasi terpilih dan alasannya (angka dari Section 7) → keterbatasan → format keluaran.)*